Given the current/past behavior of a service, can we predict whether it is moving toward a degraded/critical state in the near future?

Prediction window: next 1 hour

Target:
1 → service becomes Degrading/Critical within the next hour
0 → service remains Healthy

In [0]:
%run ./00_Project_Setup

In [0]:
## Reading Data from silver layer
from pyspark.sql import functions as F

silver_df = spark.table("`log-analytics`.silver.silver_logs")

print("Rows    :", silver_df.count())
print("Columns :", len(silver_df.columns))

In [0]:
## create time features
silver_df = (
    silver_df
    .withColumn("event_hour", F.hour("event_timestamp"))
    .withColumn("event_date", F.to_date("event_timestamp"))
)

In [0]:
## creating hourly service metrics
hourly_service_df = (
    silver_df
    .groupBy(
        "service",
        "event_date",
        "event_hour"
    )
    .agg(
        F.count("*").alias("total_logs"),
        F.sum(
            F.when(F.col("log_level") == "ERROR", 1).otherwise(0)
        ).alias("error_count"),
        F.sum(
            F.when(F.col("log_level") == "WARNING", 1).otherwise(0)
        ).alias("warning_count"),
        F.sum(
            F.when(F.col("log_level") == "CRITICAL", 1).otherwise(0)
        ).alias("critical_count"),
        F.sum("is_anomaly").alias("anomaly_count"),
        F.avg("response_time").alias("avg_response_time"),
        F.max("response_time").alias("max_response_time"),
        F.avg("severity_score").alias("avg_severity")
    )
)

In [0]:
## calculating error rate
hourly_service_df = hourly_service_df.withColumn(
    "error_rate",
    F.col("error_count") / F.col("total_logs")
)

In [0]:
display(hourly_service_df.limit(20))

In [0]:
## Create the future 1-hour target
## we need to determine what happens in the next hour for the same service

hourly_service_df = hourly_service_df.withColumn(
    "hour_timestamp",
    F.to_timestamp(
        F.concat_ws(
            " ",
            F.col("event_date").cast("string"),
            F.lpad(F.col("event_hour").cast("string"), 2, "0")
        ),
        "yyyy-MM-dd HH"
    )
)

In [0]:
hourly_service_df = hourly_service_df.withColumn(
    "next_hour_timestamp",
    F.expr("hour_timestamp + INTERVAL 1 HOUR")
)

In [0]:
## Create a future-hour dataset containing only the metrics
## we need to determine the next-hour outcome

future_df = hourly_service_df.select(
    F.col("service").alias("future_service"),
    F.col("hour_timestamp").alias("future_hour_timestamp"),
    
    # Metrics from the future hour
    F.col("total_logs").alias("future_total_logs"),
    F.col("error_count").alias("future_error_count"),
    F.col("warning_count").alias("future_warning_count"),
    F.col("critical_count").alias("future_critical_count"),
    F.col("anomaly_count").alias("future_anomaly_count"),
    F.col("error_rate").alias("future_error_rate"),
    F.col("avg_response_time").alias("future_avg_response_time"),
    F.col("max_response_time").alias("future_max_response_time"),
    F.col("avg_severity").alias("future_avg_severity"),
)

display(future_df.limit(20))

In [0]:
## match current hour to next hour
early_warning_base_df = (
    hourly_service_df.alias("current")
    .join(
        future_df.alias("future"),
        (
            (F.col("current.service") == F.col("future.future_service")) &
            (F.col("current.next_hour_timestamp") == F.col("future.future_hour_timestamp"))
        ),
        "left"
    )
)

In [0]:
display(
    early_warning_base_df.select(
        "current.service",
        "current.hour_timestamp",
        "current.next_hour_timestamp",
        "future.future_hour_timestamp",
        "future.future_error_rate",
        "future.future_anomaly_count",
        "future.future_critical_count",
        "future.future_avg_response_time"
    ).limit(20)
)

In [0]:
## creating early warning df
early_warning_base_df.printSchema()

In [0]:
print([
    c for c in early_warning_base_df.columns
    if "status" in c.lower()
    or "state" in c.lower()
    or "health" in c.lower()
])

In [0]:
display(
    early_warning_base_df.select(
        "service",
        "hour_timestamp",
        "future_hour_timestamp",
        "future_error_rate",
        "future_anomaly_count",
        "future_critical_count",
        "future_avg_response_time",
        "future_avg_severity"
    ).limit(20)
)

In [0]:
## lets create early warning target

early_warning_base_df = early_warning_base_df.withColumn(
    "early_warning_target",
    F.when(
        (
            (F.col("future_critical_count") > 0)
        ),
        1
    ).otherwise(0)
)

In [0]:
display(
    early_warning_base_df
    .groupBy("early_warning_target")
    .count()
    .orderBy("early_warning_target")
)

In [0]:
display(
    early_warning_base_df.select(
        "service",
        "hour_timestamp",
        "next_hour_timestamp",
        "future_critical_count",
        "future_error_rate",
        "future_anomaly_count",
        "future_avg_response_time",
        "early_warning_target"
    )
    .orderBy("hour_timestamp")
    .limit(30)
)